In [56]:
import baostock as bs
import pandas as pd
from tqdm import tqdm
import time
import os
import random
import akshare as ak
import time
import json
from pathlib import Path
from typing import List, Dict
import pandas as pd
import yfinance as yf
import time
import json
import math
from pathlib import Path
from typing import List, Dict, Any
import pandas as pd
import numpy as np
import yfinance as yf

In [57]:
import os
proxy = 'http://127.0.0.1:7890' # 代理设置，此处修改
os.environ['HTTP_PROXY'] = proxy
os.environ['HTTPS_PROXY'] = proxy

In [36]:
# 彻底禁掉代理（见上一节）
for v in ("HTTP_PROXY","HTTPS_PROXY","http_proxy","https_proxy"):
    os.environ.pop(v, None)

In [37]:
# ------------------ 配置 ------------------
NASDAQ_TXT = r"C:\Users\Administrator\桌面\USA_nasdaq.txt"
OTHER_TXT  = r"C:\Users\Administrator\桌面\USA_other.txt"
CACHE_FILE = "yf_fetch_cache.json"

In [38]:
# 防限流设置（可按需要调大）
SLEEP_BETWEEN = 1  # 每个请求后 sleep 秒
MAX_RETRIES = 1

# 要抓取的 info 字段（直接来自 ticker.info）
INFO_FIELDS = [
    "shortName", "longName", "exchange", "quoteType",
    "sector", "industry", "marketCap", "enterpriseValue",
    "floatShares", "sharesOutstanding", "trailingPE",
    "priceToBook", "earningsGrowth", "revenueGrowth",
    "profitMargins", "grossMargins", "operatingMargins",
    "dividendYield", "fiveYearAvgDividendYield"
]

# 要从财务表补充/校验的指标（key names we will search for）
# 这些在 yfinance.financials / quarterly_financials 表中通常使用类似的行名。
REVENUE_KEYS = ["Total Revenue", "TotalRevenue", "Revenue"]
REVENUE_PER_SHARE_KEYS = ["Revenue Per Share", "RevenuePerShare"]
# financials usually has Revenue, Gross Profit, Operating Income, Net Income
TOTAL_REVENUE_ROW_NAMES = ["Total Revenue", "TotalRevenue", "Revenue"]
GROSS_PROFIT_ROW_NAMES = ["Gross Profit", "GrossProfit"]
OPERATING_INCOME_ROW_NAMES = ["Operating Income", "OperatingIncome", "Operating Income (Loss)"]
NET_INCOME_ROW_NAMES = ["Net Income", "NetIncome", "Net Income Applicable To Common Shares"]
REVENUE_PER_SHARE_ROW_NAMES = ["Revenue Per Share", "RevenuePerShare"]

# ------------------ 辅助函数 ------------------
def read_symbols_from_txt(nasdaq_txt: str, other_txt: str) -> pd.DataFrame:
    frames = []
    if Path(nasdaq_txt).exists():
        df_n = pd.read_csv(nasdaq_txt, sep="|", dtype=str, on_bad_lines="skip")
        if "Symbol" in df_n.columns:
            df_n = df_n.rename(columns={"Symbol": "symbol", "Security Name": "security_name"})
            df_n["source"] = Path(nasdaq_txt).name
            frames.append(df_n[["symbol", "security_name", "source"]])
    if Path(other_txt).exists():
        df_o = pd.read_csv(other_txt, sep="|", dtype=str, on_bad_lines="skip")
        # otherlisted 常用列 'ACT Symbol' or 'NASDAQ Symbol'
        if "ACT Symbol" in df_o.columns:
            df_o = df_o.rename(columns={"ACT Symbol": "symbol", "Security Name": "security_name"})
        elif "NASDAQ Symbol" in df_o.columns:
            df_o = df_o.rename(columns={"NASDAQ Symbol": "symbol", "Security Name": "security_name"})
        else:
            # fallback if columns differ
            if "Symbol" in df_o.columns:
                df_o = df_o.rename(columns={"Symbol": "symbol", "Security Name": "security_name"})
        df_o["source"] = Path(other_txt).name
        frames.append(df_o[["symbol", "security_name", "source"]])
    if not frames:
        raise RuntimeError("找不到输入文件：请把 nasdaqlisted.txt / otherlisted.txt 放到当前目录。")
    combined = pd.concat(frames, ignore_index=True)
    combined["symbol"] = combined["symbol"].astype(str).str.strip()
    combined["security_name"] = combined["security_name"].astype(str).str.strip()
    combined = combined.dropna(subset=["symbol"]).drop_duplicates(subset=["symbol"]).reset_index(drop=True)
    return combined

def load_cache(path: str) -> Dict[str, Any]:
    if Path(path).exists():
        try:
            return json.loads(Path(path).read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_cache(path: str, data: Dict[str, Any]):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def safe_fetch_ticker_info(symbol: str) -> Dict[str, Any]:
    """抓取 ticker.info；尝试重试；返回 dict（可能含很多字段）"""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            t = yf.Ticker(symbol)
            info = t.info or {}
            return info
        except Exception as e:
            wait = SLEEP_BETWEEN * attempt
            print(f"[{symbol}] info fetch failed attempt {attempt}: {e}. wait {wait}s")
            time.sleep(wait)
    return {}

def safe_fetch_financial_table(ticker_obj, attr_name: str):
    """从 ticker 对象取 financials 或 quarterly_financials，返回 DataFrame 或 None"""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if attr_name == "financials":
                df = ticker_obj.financials
            elif attr_name == "quarterly_financials":
                df = ticker_obj.quarterly_financials
            else:
                return None
            # 返回 DataFrame（若为空，yfinance 可能返回空 DataFrame）
            if isinstance(df, pd.DataFrame) and not df.empty:
                return df
            else:
                return pd.DataFrame()  # empty DF
        except Exception as e:
            wait = SLEEP_BETWEEN * attempt
            print(f"[{symbol}] {attr_name} fetch failed attempt {attempt}: {e}. wait {wait}s")
            time.sleep(wait)
    return pd.DataFrame()

def find_first_row_value(df: pd.DataFrame, possible_names: List[str]):
    """在 df.index 中查找可能的行名，返回该行第一个非空值（或 NaN）"""
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return np.nan
    idx = df.index.astype(str).tolist()
    for name in possible_names:
        if name in idx:
            row = df.loc[name]
            # row could be Series with columns dates; choose first (most recent) column if numeric
            # Convert to numeric if possible, take first non-null numeric
            vals = [v for v in row.tolist() if (v is not None and (not (isinstance(v, float) and math.isnan(v))))]
            if vals:
                # return first numeric-like value (float/int), else first value
                for v in vals:
                    try:
                        return float(v)
                    except Exception:
                        return v
            else:
                return np.nan
    # try fuzzy: lower-match
    lowered_idx = [i.lower() for i in idx]
    for name in possible_names:
        for i, li in enumerate(lowered_idx):
            if name.lower() == li or name.lower() in li:
                row = df.iloc[i]
                vals = [v for v in row.tolist() if (v is not None and (not (isinstance(v, float) and math.isnan(v))))]
                if vals:
                    try:
                        return float(vals[0])
                    except Exception:
                        return vals[0]
    return np.nan

In [46]:
# ------------------ 主流程 ------------------
df_sym = read_symbols_from_txt(NASDAQ_TXT, OTHER_TXT)
symbols = df_sym["symbol"].tolist()
print(f"Total symbols: {len(symbols)}")

cache = load_cache(CACHE_FILE)
results = []

for idx, row in df_sym.iterrows():
    symbol = row["symbol"]
    src_name = row.get("security_name", "")
    source_file = row.get("source", "")
    # 如果缓存存在，直接用缓存结果
    if symbol in cache:
        rec = cache[symbol]
        print(f"[{symbol}] loaded from cache")
        results.append(rec)
        continue

    print(f"[{idx+1}/{len(symbols)}] Fetching {symbol} ...")
    # 拉 info
    info = safe_fetch_ticker_info(symbol)
    # yfinance Ticker object for financial tables
    t = yf.Ticker(symbol)

    # 拉年报和季报（DataFrame）
    fin_annual = safe_fetch_financial_table(t, "financials")         # 年度（columns: recent years）
    fin_quarterly = safe_fetch_financial_table(t, "quarterly_financials")  # 季度

    # 从 info 拿基础字段（原样取）; 若不存在则为 None -> 转 np.nan
    rec: Dict[str, Any] = {"symbol": symbol, "security_name_file": src_name, "source_file": source_file}
    for f in INFO_FIELDS:
        # 将不存在或空值统一为 None (later convert to np.nan)
        val = info.get(f, None)
        rec[f] = val if val is not None else None

    # 从财报表里尽可能补 totalRevenue / revenuePerShare / totalRevenue (annual) 和 quarterly
    # 年度 Total Revenue
    total_revenue_annual = find_first_row_value(fin_annual, TOTAL_REVENUE_ROW_NAMES)
    total_revenue_quarterly = find_first_row_value(fin_quarterly, TOTAL_REVENUE_ROW_NAMES)

    # revenue per share
    revenue_per_share_annual = find_first_row_value(fin_annual, REVENUE_PER_SHARE_ROW_NAMES)
    revenue_per_share_quarterly = find_first_row_value(fin_quarterly, REVENUE_PER_SHARE_ROW_NAMES)

    # gross profit, operating income
    gross_profit_annual = find_first_row_value(fin_annual, GROSS_PROFIT_ROW_NAMES)
    gross_profit_quarterly = find_first_row_value(fin_quarterly, GROSS_PROFIT_ROW_NAMES)

    operating_income_annual = find_first_row_value(fin_annual, OPERATING_INCOME_ROW_NAMES)
    operating_income_quarterly = find_first_row_value(fin_quarterly, OPERATING_INCOME_ROW_NAMES)

    # 收拢到 rec，注意用括号备注年/季
    rec["totalRevenue (Annual)"] = (None if total_revenue_annual is None or (isinstance(total_revenue_annual, float) and math.isnan(total_revenue_annual)) else total_revenue_annual)
    rec["totalRevenue (Quarterly)"] = (None if total_revenue_quarterly is None or (isinstance(total_revenue_quarterly, float) and math.isnan(total_revenue_quarterly)) else total_revenue_quarterly)

    rec["revenuePerShare (Annual)"] = (None if revenue_per_share_annual is None or (isinstance(revenue_per_share_annual, float) and math.isnan(revenue_per_share_annual)) else revenue_per_share_annual)
    rec["revenuePerShare (Quarterly)"] = (None if revenue_per_share_quarterly is None or (isinstance(revenue_per_share_quarterly, float) and math.isnan(revenue_per_share_quarterly)) else revenue_per_share_quarterly)

    rec["grossProfit (Annual)"] = (None if gross_profit_annual is None or (isinstance(gross_profit_annual, float) and math.isnan(gross_profit_annual)) else gross_profit_annual)
    rec["grossProfit (Quarterly)"] = (None if gross_profit_quarterly is None or (isinstance(gross_profit_quarterly, float) and math.isnan(gross_profit_quarterly)) else gross_profit_quarterly)

    rec["operatingIncome (Annual)"] = (None if operating_income_annual is None or (isinstance(operating_income_annual, float) and math.isnan(operating_income_annual)) else operating_income_annual)
    rec["operatingIncome (Quarterly)"] = (None if operating_income_quarterly is None or (isinstance(operating_income_quarterly, float) and math.isnan(operating_income_quarterly)) else operating_income_quarterly)

    # Note: some metrics like earningsGrowth, revenueGrowth, profitMargins etc. we already attempted to take from info
    # Keep the values as-is (likely present in info)
    # Make sure numeric -> if missing set None

    # 标准化 None -> None (pandas -> NaN later)
    # Save into cache and results
    cache[symbol] = rec
    results.append(rec)

    # 定期写缓存，避免中断重跑
    if (idx + 1) % 50 == 0:
        save_cache(CACHE_FILE, cache)
        print(f"Saved cache at symbol {symbol}")

    time.sleep(SLEEP_BETWEEN)

# 最后保存缓存
save_cache(CACHE_FILE, cache)
# Build DataFrame
df_out = pd.DataFrame(results)

# 将 None 值统一为 np.nan（便于 CSV/后续处理）
df_out = df_out.replace({None: np.nan})

# 重新组织列顺序并增加中文括注（英文在前，中文在后）
col_map = {
    "symbol": "symbol (代码)",
    "security_name_file": "security_name_in_file (文件中的证券名)",
    "shortName": "shortName (简称)",
    "longName": "longName (全称)",
    "exchange": "exchange (交易所)",
    "quoteType": "quoteType (报价类型)",
    "sector": "sector (行业大类)",
    "industry": "industry (细分行业)",
    "marketCap": "marketCap (市值)",
    "enterpriseValue": "enterpriseValue (企业价值)",
    "floatShares": "floatShares (流通股数)",
    "sharesOutstanding": "sharesOutstanding (总股本)",
    "trailingPE": "trailingPE (滚动市盈率)",
    "priceToBook": "priceToBook (市净率)",
    "earningsGrowth": "earningsGrowth (利润增长率)",
    "revenueGrowth": "revenueGrowth (营收增长率)",
    "profitMargins": "profitMargins (利润率)",
    "grossMargins": "grossMargins (毛利率)",
    "operatingMargins": "operatingMargins (营业利润率)",
    "totalRevenue (Annual)": "totalRevenue (Annual) (总营收 (年))",
    "totalRevenue (Quarterly)": "totalRevenue (Quarterly) (总营收 (季))",
    "revenuePerShare (Annual)": "revenuePerShare (Annual) (每股营收 (年))",
    "revenuePerShare (Quarterly)": "revenuePerShare (Quarterly) (每股营收 (季))",
    "grossProfit (Annual)": "grossProfit (Annual) (毛利润 (年))",
    "grossProfit (Quarterly)": "grossProfit (Quarterly) (毛利润 (季))",
    "operatingIncome (Annual)": "operatingIncome (Annual) (营业利润 (年))",
    "operatingIncome (Quarterly)": "operatingIncome (Quarterly) (营业利润 (季))",
    "dividendYield": "dividendYield (股息率)",
    "fiveYearAvgDividendYield": "fiveYearAvgDividendYield (5年平均股息率)",
    "source_file": "source_file (来源文件)"
}

# Ensure all mapped columns exist in df_out
final_cols = []
for k, v in col_map.items():
    if k in df_out.columns:
        final_cols.append(k)
    else:
        # add column with NaN if missing (keeps consistent schema)
        df_out[k] = np.nan
        final_cols.append(k)

# Reorder and rename
df_out = df_out[final_cols].rename(columns=col_map)

# Save CSV


print("Preview:")
print(df_out.head(10))
print("Counts of non-null marketCap:", df_out["marketCap (市值)"].notna().sum())


In [ ]:
symbols = df_out["symbol (代码)"].dropna().unique().tolist()
print("股票数量：", len(symbols))

# ----------------------------------------
# 2. 准备拉取每日收盘价
# ----------------------------------------
START = "2025-01-01"
END   = "2025-12-09"

all_prices = []   # 存储每支股票的 DataFrame

# 防限流
sleep_seconds = 0.1

for i, sym in enumerate(symbols, 1):
    print(f"[{i}/{len(symbols)}] Fetching {sym} ...")

    try:
        df = yf.download(
            sym,
            start=START,
            end=END,
            interval="1d",
            progress=False
        )

        # 若成功获取且非空
        if not df.empty:
            # 只保留 Close 列，并改名为股票代码
            df = df[["Close"]].rename(columns={"Close": sym})
            all_prices.append(df)
        else:
            # 创建空列避免丢失
            empty_df = pd.DataFrame({sym: np.nan}, index=[pd.Timestamp(START)])
            all_prices.append(empty_df)

    except Exception as e:
        print(f"Error fetching {sym}: {e}")
        empty_df = pd.DataFrame({sym: np.nan}, index=[pd.Timestamp(START)])
        all_prices.append(empty_df)

    time.sleep(sleep_seconds)

# ----------------------------------------
# 3. 合并所有股票的 Close 成一个总 df
# ----------------------------------------
df_prices = pd.concat(all_prices, axis=1)

# 日期排序
df_prices = df_prices.sort_index()

# print("价格数据预览：")
# print(df_prices.head())
# print(df_prices.tail())

# 保存
df_prices.to_csv('../data/USA_stock.csv',encoding='utf-8-sig',index=False)

[1456/12076] Fetching EGGQ ...


C:\Users\Administrator\AppData\Local\Temp\ipykernel_33524\2273112668.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(


[1457/12076] Fetching EGHA ...


C:\Users\Administrator\AppData\Local\Temp\ipykernel_33524\2273112668.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
